# PMS assistant: free LoRA fine-tune (Colab T4, Unsloth)
Free tier only. Runtime: GPU (T4). Trains a small LoRA adapter on `train_sft.jsonl` (synthetic eval data) and checks it on `heldout_questions.jsonl`.
Result: `adapter_model.safetensors` + `adapter_config.json` that Cloudflare Workers AI can serve (base `@cf/mistralai/mistral-7b-instruct-v0.2-lora`).

Read first: the question bank is built from 25 synthetic records. It teaches phrasing and answer style, not real numbers. The assistant must keep reading real numbers from the PMS tools; do not let the model answer from memory.

In [ ]:
!pip install -q unsloth

In [ ]:
BASE = "unsloth/mistral-7b-instruct-v0.2-bnb-4bit"   # must match the Workers AI base model family (Mistral 7B Instruct v0.2)
RAW = "https://raw.githubusercontent.com/mustafapato863-netizen/PMS/main/evaluation/pms-ai/question_bank/"
import urllib.request
for f in ("train_sft.jsonl", "heldout_questions.jsonl"):
    urllib.request.urlretrieve(RAW + f, f)

In [ ]:
from unsloth import FastLanguageModel
model, tok = FastLanguageModel.from_pretrained(BASE, max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=16, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
)

In [ ]:
import json
from datasets import Dataset
rows = [json.loads(l) for l in open("train_sft.jsonl")]
# Mistral v0.2 has no system role: fold the system line into the user turn.
def fmt(r):
    m = r["messages"]; sys_, user, asst = m[0]["content"], m[1]["content"], m[2]["content"]
    return {"text": tok.apply_chat_template([{"role":"user","content":sys_+"\n\n"+user},{"role":"assistant","content":asst}], tokenize=False)}
ds = Dataset.from_list(rows).map(fmt, remove_columns=["messages","meta"])
print(len(ds))

In [ ]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(model=model, tokenizer=tok, train_dataset=ds,
    args=SFTConfig(per_device_train_batch_size=4, gradient_accumulation_steps=4, num_train_epochs=2,
                   learning_rate=2e-4, logging_steps=10, output_dir="out", dataset_text_field="text",
                   max_seq_length=1024, fp16=True, report_to="none"))
trainer.train()

In [ ]:
# Quick held-out check: print answers for 15 unseen phrasings, compare by eye with questions.jsonl expected facts.
FastLanguageModel.for_inference(model)
held = [json.loads(l) for l in open("heldout_questions.jsonl")][:15]
for h in held:
    p = tok.apply_chat_template([{"role":"user","content":"You are the PMS performance assistant.\n\n"+h["prompt"]}], tokenize=False, add_generation_prompt=True)
    ids = tok(p, return_tensors="pt", add_special_tokens=False).to("cuda")
    out = model.generate(**ids, max_new_tokens=80)
    print(h["prompt"], "=>", tok.decode(out[0][ids["input_ids"].shape[1]:], skip_special_tokens=True), "\n")

In [ ]:
model.save_pretrained("adapter")   # writes adapter_config.json + adapter_model.safetensors
import json
cfg = json.load(open("adapter/adapter_config.json")); cfg["model_type"] = "mistral"
json.dump(cfg, open("adapter/adapter_config.json","w"), indent=2)
!ls -la adapter && du -sh adapter

In [ ]:
from google.colab import files
!cd adapter && zip -r ../pms-adapter.zip adapter_config.json adapter_model.safetensors
files.download("pms-adapter.zip")

Next (see README.md): upload the adapter to Workers AI with wrangler, A/B it against the current model on the held-out set, and only then consider routing traffic to it.